# Sage Cinema bug audit

**Audit date:** 2026-09-12  
**Scope:** active Next.js app, client hooks, API routes, media playback, PWA cache, and deployment configuration. The `old_assets/` tree is treated as stale because it is excluded from lint/tests and is not imported by the current app.

This notebook records the current behavior and the bugs/risk areas found during a static audit. It does not modify the application code.

## Executive summary

The app compiles and the existing unit tests pass, but the current production behavior has several high-impact issues:

1. The production service worker caches every same-origin GET, including API responses. This can freeze search, collections, analytics, health checks, and video-source responses until the service-worker cache is replaced.
2. Clean-source subtitles are sent through the media proxy instead of the subtitle proxy, so SRT conversion and the subtitle-host restriction are bypassed. Captions from that source path are likely unusable.
3. The media proxy does not forward `Range` requests, which can break or severely degrade seeking and large-file playback.
4. The visible server picker is not connected to unified-source selection, exposes only the first two configured servers, and the returned fallback embed URL is never rendered.
5. Movie/TV routing is inferred with `slug.includes('tv')`; a movie title containing `tv` can be loaded as a TV show.
6. Async search and detail requests can resolve out of order and overwrite newer state.
7. The current lint command fails with 9 errors, even though TypeScript, build, and the 7 existing tests pass.

## Architecture map

```text
Browser client
  ├─ Home / genre / detail pages
  │    ├─ localStorage/sessionStorage state
  │    └─ fetch('/api/*')
  ├─ PWA service worker (production)
  │    └─ cache-first handling for every same-origin GET
  └─ UnifiedPlayer
       └─ signed /api/media-proxy URLs

Next route handlers
  ├─ TMDB proxy routes
  ├─ analytics storage
  ├─ unified-source resolver
  └─ media/subtitle proxy routes
```

The most important systemic boundary is the service worker: it sits below the client fetch code and can override `cache: 'no-store'` by manually returning/storing Cache API entries.

## Verification baseline

| Check | Result | Meaning |
|---|---:|---|
| `npm run build` | PASS | Production bundle compiles and static generation completes. |
| `npx tsc --noEmit` | PASS | TypeScript does not find a type error. |
| `npm test -- --run` | PASS | 1 file / 7 tests pass; coverage is narrow. |
| `npm run lint` | FAIL | 9 errors and 5 warnings remain. |

## Confirmed findings

### 1. Service worker caches dynamic APIs (P0/P1)

Evidence: [`public/service-worker.js`](public/service-worker.js#L20-L45). The fetch handler intercepts every same-origin GET. It checks the cache first, then stores every successful response, without excluding `/api/`, media, subtitles, health, or analytics URLs and without honoring `Cache-Control: no-store`.

Impact: a production user can receive stale collection/search results, stale analytics counters, stale server health, or an expired video-source response. The cache name is `sage-cinema-shell-v1`, so these entries persist until a new service-worker cache version is deployed or the user clears site data.

Repair direction: bypass `/api/` and media routes entirely, or implement explicit network-first/runtime strategies that respect response cache directives. Version the cache when shell behavior changes.

### 2. Resolver subtitles use the wrong proxy endpoint (P1)

Evidence: [`lib/unifiedSources.ts`](lib/unifiedSources.ts#L238-L248) calls `buildMediaProxyUrl(url)` for resolver subtitles. The file already contains `buildSubtitleProxyUrl()` at lines 144–149, and the online SubDL route correctly uses it.

The media proxy returns the upstream body as-is. The subtitle proxy converts SRT to WebVTT and only permits `dl.subdl.com`. A `<track>` normally needs WebVTT, so resolver captions can fail even when the source list succeeds.

Repair direction: use `buildSubtitleProxyUrl(url)` for subtitle entries and validate/normalize every source URL before signing it.

### 3. Media proxy drops byte-range requests (P1)

Evidence: [`app/api/media-proxy/route.js`](app/api/media-proxy/route.js#L63-L86). The upstream request sends only fixed headers; it does not forward the incoming `Range` header. The response copies `Accept-Ranges`, `Content-Range`, and `ETag`, but not `Content-Length` and not the request range semantics.

Impact: MP4 playback can download the full object instead of serving byte ranges, and seeking may fail or become slow. This is especially costly for large media files.

Repair direction: forward `Range` and relevant conditional headers, preserve upstream status/length/range headers, and test seek behavior in Chromium/Safari.

### 4. Server selection is disconnected from unified playback (P1)

Evidence: [`app/movie/[id]/[slug]/page.tsx`](<app/movie/[id]/[slug]/page.tsx>#L46-L46) exposes only `VIDEO_SERVERS.slice(0, 2)`. The page passes the selected server to the API, but [`app/api/video-sources/[type]/[id]/route.js`](<app/api/video-sources/[type]/[id]/route.js>#L31-L49) calls `resolveUnifiedSources()` without passing the selected server. [`lib/unifiedSources.ts`](lib/unifiedSources.ts#L39-L44) always queries all four resolver paths.

Impact: switching Server 1/Server 2 reloads the same unified source pool, while seven configured `VIDEO_SERVERS` are inaccessible. The API also returns `fallbackEmbedURL`, but the detail page contains no iframe/embed rendering path for it; when clean sources fail, the user only sees an error.

Repair direction: choose one explicit model: either expose resolver pools as the selectable sources, or use the configured embed servers and actually render the fallback. Keep health checks and visible options derived from the same list.

### 5. Movie/TV type detection is substring-based (P1)

Evidence: [`app/movie/[id]/[slug]/page.tsx`](<app/movie/[id]/[slug]/page.tsx>#L111-L116) uses `slug?.includes('tv')` to choose the TMDB endpoint. All generated links already contain a type prefix such as `movie-...` or `tv-...`.

Impact: a movie whose title slug contains `tv` can be fetched as a TV show. The route then renders series controls or requests the wrong TMDB resource.

Repair direction: parse the prefix (`slug.startsWith('tv-')`) or pass the media type as a validated route segment/param. Never infer it from arbitrary title text.

### 6. Detail-page requests can overwrite newer navigation (P1)

Evidence: the effect in [`app/movie/[id]/[slug]/page.tsx`](<app/movie/[id]/[slug]/page.tsx>#L111-L177) starts detail and recommendation requests but has no `AbortController` or active-request guard. State updates such as `setMovie(data)`, `setSimilarMovies(...)`, `setError(...)`, and `setIsLoading(false)` are unconditional.

Impact: navigating quickly from title A to title B can allow A's slower response to replace B's title or recommendations.

Repair direction: abort the old request and guard every async continuation with a request id or `active` flag. Apply the same pattern to genre-page loading.

### 7. Search responses can arrive out of order (P1)

Evidence: [`lib/hooks/useSearch.ts`](lib/hooks/useSearch.ts#L17-L49) cancels only the debounce timer. Once `fetch()` starts, the effect has no abort signal and no query identity check before `setResults(searchResults)`.

Impact: typing quickly can display results for an older query after the input already contains a newer query.

Repair direction: use an `AbortController` per effect and ignore `AbortError`, or keep a monotonically increasing request id and only commit the latest response.

### 8. HTTP API failures are rendered as empty success states (P1)

Evidence: [`app/page.tsx`](app/page.tsx#L323-L347) calls `response.json()` without checking `response.ok`, then maps missing `results` to empty arrays. The same pattern appears in [`app/genre/[id]/page.tsx`](<app/genre/[id]/page.tsx>#L57-L74), [`lib/context/AppContext.tsx`](lib/context/AppContext.tsx#L42-L59), and [`lib/hooks/useSearch.ts`](lib/hooks/useSearch.ts#L36-L46).

Impact: a 500/503 from TMDB, analytics, or the deployment is often shown as an empty shelf or zero results rather than an actionable error. Missing `TMDB_API_KEY` is especially misleading.

Repair direction: centralize a `fetchJson` helper that checks `response.ok`, validates the response shape, and exposes a visible retry/error state.

### 9. Search relevance regex accepts raw user input (P1)

Evidence: [`app/api/search/route.js`](app/api/search/route.js#L150-L157) constructs `new RegExp('\\b' + q + '\\b', 'i')` without escaping `q`.

Impact: punctuation-heavy queries such as `C++` throw `SyntaxError`, the route falls into its 500 handler, and the client displays no results. Other characters such as `+`, `*`, `?`, `[`, and `(` also change matching semantics.

Repair direction: escape regex metacharacters or use non-regex matching for the word-boundary case.

### 10. Unified resolver title is double-encoded (P1/P2)

Evidence: [`lib/unifiedSources.ts`](lib/unifiedSources.ts#L186-L203) first applies `encodeURIComponent(options.title)` and then inserts that value with `URLSearchParams.set()`, which encodes percent signs again.

Impact: `The Matrix: Reloaded` is sent as `The%2520Matrix%253A%2520Reloaded` on the wire. A resolver relying on title matching receives percent escapes instead of normal text; TMDB ID may hide the defect for some titles.

Repair direction: pass the raw title to `URLSearchParams.set()` and let the URL builder encode it once.

### 11. Failed detail fetches are cached permanently for the session (P1/P2)

Evidence: [`lib/moviePrefetch.ts`](lib/moviePrefetch.ts#L4-L20) stores the promise in `detailsCache`; the promise converts any non-OK response or network error to `null`, and the failed promise is never removed.

Impact: a transient TMDB/network failure during hover prefetch can make the same title look permanently unavailable until a full page reload.

Repair direction: evict rejected/null requests or cache only successful responses with a short TTL.

### 12. Analytics fallback is not durable on serverless deployments (P1/P2)

Evidence: [`lib/analyticsStore.ts`](lib/analyticsStore.ts#L14-L15) writes fallback data to `.local-data/site-stats.json`; [`getRedis()`](lib/analyticsStore.ts#L40-L55) uses local storage when Redis variables are absent. The deployment notes identify Netlify CDN behavior, while `.local-data/` is gitignored.

Impact: without a configured Redis backend, serverless instances can lose counters between executions/deployments or maintain fragmented per-instance counters.

Repair direction: require/configure a durable hosted store in production, or explicitly disable analytics instead of silently using a local file fallback.

### 13. Advertised platform search coverage is incomplete (P2)

Evidence: [`lib/streamingServices.ts`](lib/streamingServices.ts#L17-L96) advertises Paramount+, Hulu, Warner, Marvel, Universal, Sony, and A24, but [`app/api/search/route.js`](app/api/search/route.js#L24-L34) only maps Vivamax, Netflix, HBO, Disney, Apple, and Amazon.

Impact: platform tiles/labels can lead users to generic title search rather than a provider/company catalog, and results may be empty or unrelated.

Repair direction: maintain one shared, validated platform mapping or remove unsupported brands from the UI.

### 14. Genre search platform logos are malformed (P2)

Evidence: `SEARCH_BRANDS[*].logoPath` already contains a full TMDB URL, but [`components/SearchModal.tsx`](components/SearchModal.tsx#L164-L171) prepends `https://image.tmdb.org/t/p/w92` again.

Impact: platform quick-link images resolve to a URL shaped like `.../w92https://image.tmdb.org/...`, so those logos fail in the SearchModal path.

Repair direction: use the path-or-URL normalization already used by `ServiceBottomNav`, or store only relative paths.

### 15. APK works locally but is ignored by Git (P1 when deploying from the repo)

Evidence: `public/sagemovies-latest.apk` exists in this workspace but [`.gitignore`](.gitignore#L35-L35) ignores `public/*.apk`, and the file is not tracked. [`app/api/app-version/route.ts`](app/api/app-version/route.ts#L3-L8) returns a same-origin URL for it.

Impact: a fresh deployment from version control will not contain the APK unless the deployment pipeline separately uploads it, so the advertised download URL returns 404.

Repair direction: use a tracked/external artifact URL or add an explicit deployment upload step and verify the final URL in CI.

## Code-quality failures currently visible

`npm run lint` currently fails with 9 errors and 5 warnings. The errors include:

- unescaped apostrophes in `app/genre/[id]/page.tsx` and `components/SeeAllModal.tsx`;
- synchronous state updates inside effects in `components/DownloadAppModal.tsx`, `components/MaintenancePage.tsx`, `lib/context/AppContext.tsx`, `lib/hooks/useSearch.ts`, and `lib/hooks/useWatchHistory.ts`.

The warnings include the missing `fetchMore` dependency in `SeeAllModal`, a raw `<img>` in `ServiceBottomNav`, and anonymous config exports. These are not the root causes of the media bugs above, but they make regressions harder to catch.

## Reproducible micro-checks

The following examples demonstrate two deterministic defects without calling external services.

In [ ]:
# JavaScript's RegExp is the runtime used by the route. Raw C++ throws there.
import subprocess
node_script = r"""
const q = 'C++';
try { new RegExp('\\b' + q + '\\b', 'i'); }
catch (error) { console.log('Raw search query raises:', error.message); }
"""
print(subprocess.run(['node', '-e', node_script], capture_output=True, text=True).stdout.strip())

# URLSearchParams behaves like this: pre-encoding before setting a parameter
# turns %20 into %2520 on the wire.
from urllib.parse import urlencode
print('Double-encoded query:', urlencode({'title': 'The%20Matrix%3A%20Reloaded'}))
print('Correctly encoded query:', urlencode({'title': 'The Matrix: Reloaded'}))

Expected output includes a regex error for `C++`, plus `title=The%2520Matrix%253A%2520Reloaded` for the double-encoded case.

## Recommended repair order

1. Fix the service-worker routing first; it can mask every other fix in production.
2. Fix media proxy range handling and subtitle proxy selection, then test real MP4/HLS playback and captions.
3. Make media type and source-server selection explicit; either render fallback embeds or remove the misleading fallback contract.
4. Add cancellation/request identity to search, detail, and genre loads.
5. Check `response.ok` and response shapes in one shared client helper.
6. Escape search input, remove double encoding, and evict failed detail-cache entries.
7. Make analytics and APK delivery deployment-safe.
8. Clear the lint errors and add tests for the above regressions.

## Suggested regression tests

- Service worker: `/api/search`, `/api/video-sources`, `/api/analytics`, and `/api/video-health` must bypass the shell cache.
- Search: punctuation queries (`C++`, `A+B`, `[abc]`) must return a normal response.
- Resolver URL construction: title is encoded exactly once.
- Media proxy: incoming `Range: bytes=...` reaches upstream and returns 206/range headers.
- Subtitles: resolver SRT is returned as WebVTT through `/api/subtitles/file`.
- Routing: a movie slug containing `tv` still uses `type=movie`; only `tv-` uses TV.
- Async hooks: an older response cannot replace newer query/title state.
- Deployment smoke checks: `GET /sagemovies-latest.apk` and configured analytics persistence work on a clean deployment.